# Entrenamiento y selección del champion

Este notebook ejecuta una corrida controlada. Compara DummyClassifier, LogisticRegression y RandomForest, cada uno con y sin PageValues; selecciona por F1 de validación, evalúa test una sola vez, registra siete runs en MLflow y exporta el pipeline completo.

In [1]:
import os
import subprocess
from pathlib import Path

from online_shoppers.data import load_dataset
from online_shoppers.training import train_champion

ROOT = Path.cwd() if (Path.cwd() / 'pyproject.toml').exists() else Path.cwd().parent
DATA_PATH = ROOT / 'data/raw/online_shoppers_intention.csv'
MODEL_PATH = ROOT / 'models/champion.joblib'
METADATA_PATH = ROOT / 'models/model_metadata.json'
METRICS_PATH = ROOT / 'reports/model_metrics.json'
TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', f"sqlite:///{ROOT / 'mlflow.db'}")
try:
    GIT_REVISION = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    GIT_REVISION = 'local'
sessions = load_dataset(DATA_PATH)
sessions.shape

/Users/adrianalarcon/Documents/uniandes/maia_despliegue_soluciones_microproyecto/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(12330, 18)

In [2]:
outcome = train_champion(
    sessions,
    artifact_path=MODEL_PATH,
    metadata_path=METADATA_PATH,
    metrics_path=METRICS_PATH,
    tracking_uri=TRACKING_URI,
    random_seed=42,
    forest_estimators=300,
    git_revision=GIT_REVISION,
)
outcome

2026/08/12 22:40:57 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/12 22:40:57 INFO mlflow.store.db.utils: Updating database tables


2026/08/12 22:40:57 INFO mlflow.tracking.fluent: Experiment with name 'online-shoppers-purchase-intention' does not exist. Creating a new experiment.


TrainingOutcome(champion_name='random_forest', include_page_values=True, threshold=0.6023926908169313, validation_metrics={'roc_auc': 0.9246121213265609, 'pr_auc': 0.7163752172405071, 'precision': 0.6590909090909091, 'recall': 0.6850393700787402, 'f1': 0.6718146718146718, 'brier_score': 0.08856033326850907, 'true_negative': 1950, 'false_positive': 135, 'false_negative': 120, 'true_positive': 261}, test_metrics={'roc_auc': 0.9275230878998302, 'pr_auc': 0.7196639496497541, 'precision': 0.6567164179104478, 'recall': 0.6910994764397905, 'f1': 0.673469387755102, 'brier_score': 0.08683937689952939, 'true_negative': 1946, 'false_positive': 138, 'false_negative': 118, 'true_positive': 264})

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

metrics_payload = json.loads(METRICS_PATH.read_text())
candidate_metrics = pd.DataFrame(metrics_payload['candidates']).T[['f1', 'pr_auc', 'roc_auc']]
display(candidate_metrics.sort_values('f1', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
candidate_metrics.sort_values('f1').plot.barh(ax=axes[0])
axes[0].set(title='Métricas de validación por candidato', xlabel='Métrica', ylabel='')
test = metrics_payload['test']
confusion = [[test['true_negative'], test['false_positive']], [test['false_negative'], test['true_positive']]]
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1])
axes[1].set(title='Matriz de confusión del champion en test', xlabel='Predicción', ylabel='Real')
axes[1].set_xticklabels(['No compra', 'Compra'])
axes[1].set_yticklabels(['No compra', 'Compra'], rotation=0)
fig.tight_layout()
fig.savefig(ROOT / 'reports/figures/model_comparison.png', dpi=150)
plt.show()

Abra la evidencia local con `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db`. El joblib solo debe cargarse desde este flujo controlado; pickle/joblib no es seguro para archivos suministrados por terceros.